## 3.3 Die Cholesky-Zerlegung für positiv definite Matrizen

**Implementierung 3.5: Direkte Berechnung der Cholesky-Zerlegung**

In [1]:
import numpy as np 

def cholesky(A):
    n, m = A.shape
    assert n == m, "Matrix muss quadratisch sein!"
    L = np.zeros_like(A)
    
    for j in range(0, n):
        l = 0
        for k in range(0, j):
            l += L[j, k]**2
        L[j, j] = np.sqrt(A[j, j] - l)
        
        for i in range(j + 1, n):
            l = 0
            for k in range(j):
                l += L[i, k] * L[j, k]
            L[i, j] = (A[i, j] - l) / L[j, j]
    return L

*Ergänzende Einzelheiten zum Code*
- Mit `A.shape` erhalten wir ein `tupel` der Dimensionen von `A`. Wir gehen somit an dieser Stelle davon aus, dass der Funktion eine quadratische Matrix übergeben wurde.
- Mit `np.zeros_like(A)` erstellen wir eine neue, mit 0 Einträgen gefüllte Matrix, welche dieselben Eigenschaften wie `A` hat.

Wir wenden nun unsere Implementierung auf folgende Matrix an

In [2]:
A = np.array([[1, 1,  1],
              [1, 4, 1],
              [1, 1, 9]], dtype=np.half)
L = cholesky(A)

print(L)

[[1.    0.    0.   ]
 [1.    1.732 0.   ]
 [1.    0.    2.828]]


Mit der Implementierung des Cholesky-Verfahrens von `numpy` erhalten wir

In [3]:
np.linalg.cholesky(A.astype(np.double))

array([[1.        , 0.        , 0.        ],
       [1.        , 1.73205081, 0.        ],
       [1.        , 0.        , 2.82842712]])

Um hiermit ein System lösen zu können, müssen wir das Vorwärtseinsetzen mit nicht-1-Einträgen auf der Diagonalen implementieren. 

In [4]:
def vorwaerts_einsetzen(L, b):
    x = np.zeros_like(b)

    for i in range(0, b.shape[0]):
        xr = 0
        for j in range(0, i):
            xr += L[i, j] * x[j]
        x[i] = (b[i] - xr) / L[i, i]
    return x

In [5]:
def rueckwaerts_einsetzen(R, b):
    assert (len(b.shape) == 1), 'Rechte seite ist kein Vektor'
    n = b.shape[0]
    assert (R.shape == (n, n)), 'Matrix hat falsche Dimensionen'

    x = np.empty_like(b)

    x[n - 1] = b[n - 1] / R[n - 1, n - 1]
    for i in range(n - 2, -1, -1):
        xr = 0
        for j in range(i + 1, n):
            xr += R[i, j] * x[j]
        x[i] = (b[i] - xr) / R[i, i]
    return x

Mit der rechten Seite
$$
b = \begin{pmatrix}3\\6\\11\end{pmatrix}
$$
Ist die Lösung gegeben durch
$$
x = \begin{pmatrix}1\\1\\11\end{pmatrix}
$$

In [6]:
b = np.array([3, 6, 11], dtype=np.half)
x_ex = np.array([1., 1., 1.])

Mit unserer Cholesky-Zerlegung und unter Verwendung von `half` Gleitkommazahlen erhalten wir dann

In [7]:
y = vorwaerts_einsetzen(L, b)
x = rueckwaerts_einsetzen(L.transpose(), y)
print(x)

[1.     0.9995 1.    ]


Und erhalten den relativen Fehler

In [8]:
np.linalg.norm(x - x_ex) / np.linalg.norm(x_ex)

np.float64(0.000281909311127747)

Da diese Matrix recht gut konditioniert ist ($cond(A)\approx 14$), bekommen wir bei `single` oder `double` Gleitkommazahlen sogar die exakte Lösung. Probieren Sie es aus!

## Programmierabgabe

**Gruppenmitglieder von Gruppe 27**

Name, Matrikel Nummer <br>
Artur Schönfisch, 10081708 <br>
Python Version: 3.13.7  <br>

In [164]:
import sys
print(sys.version)

3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]


In [10]:
# Aufgabe 4.5b)
def factorize_cholesky_tridiagonal(a,b):
    # Annahme: A symmetrisch positiv definit, sonst wird der Radikand negativ.
    n = len(b)
    l = np.zeros(n - 1)
    c = np.zeros(n)

    c[0] = np.sqrt(b[0])
    for i in range(n - 1):
        l[i] = a[i]/c[i]
        c[i+1] = np.sqrt(b[i+1] - l[i]**2)
    return l, c

In [11]:
# Aufgabe 4.5c)
def solve_triadiagonal_system(a, b, f):
    n = len(f)
    l, c = factorize_cholesky_tridiagonal(a, b)

    # Vorwärtseinsetzen: L y = f
    y = np.zeros(n)
    y[0] = f[0]/c[0]
    for i in range(1, n):
        y[i] = (f[i] - l[i-1]*y[i-1])/c[i]

    # Rückwärtseinsetzen: L^T x = y
    x = np.zeros(n)
    x[-1] = y[-1]/c[-1]
    for i in range(n - 2, -1, -1):
        x[i] = (y[i] - l[i]*x[i+1])/c[i]
    return x

In [12]:
# Aufgabe 4.5d)
n = 1000
b = np.full(n, 2.0)
a = np.full(n-1, -1.0)
f = np.full(n, -9.81)
b[0] = 1
b[-1] = 1
a[0] = 0
a[-1] = 0
f[0] = 0
f[-1] = 0

x = solve_triadiagonal_system(a, b, f)

Ax = b * x               # Diagonale
Ax[1:]  += a * x[:-1]    # links der Diagonale
Ax[:-1] += a * x[1:]     # rechts der Diagonale
residuum = np.linalg.norm(Ax - f, ord=np.inf)

print("Residuum ||Ax - f||_inf:", residuum)

Residuum ||Ax - f||_inf: 6.426130738645952e-10


In [13]:
# Aufgabe 4.5e)
def vectors_to_tridiagonal_matrix(a, b, c):
    # b auf die Hauptdiagonale, a darunter, c darüber
    A = np.diag(b, 0) + np.diag(a, -1) + np.diag(c, 1)
    return A

In [14]:
# Aufgabe 4.5f)
n = 10
b = np.full(n, 2.0)
a = np.full(n-1, -1.0)
zeroes = np.full(n-1, 0.0)
b[0] = 1
b[-1] = 1
a[0] = 0
a[-1] = 0

l, c = factorize_cholesky_tridiagonal(a, b)
L = vectors_to_tridiagonal_matrix(l, c, zeroes)   # l unter der Diagonale, c auf der Diagonale, darüber Nullen
B = np.matmul(L, L.T)
print(B)

# Vergleich mit der Matrix aus (1)
A = vectors_to_tridiagonal_matrix(a, b, a)
print("B gleich A:", np.allclose(B, A))

[[ 1.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  2. -1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0. -1.  2. -1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  2. -1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. -1.  2. -1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0. -1.  2. -1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0. -1.  2. -1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0. -1.  2. -1.  0.]
 [ 0.  0.  0.  0.  0.  0.  0. -1.  2.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  1.]]
B gleich A: True
